# 01 — PDF / Paper / Code Extraction

Downloads and extracts text from verified open-access RF engineering resources.

## Tier 1 — Highest Priority (RF-specific)
- **Chipster** — adeirman46's own PySpice analog circuit generator + notebooks
- **AnalogCoder** — AAAI 2025 Oral: 24 analog circuit benchmark problems, PySpice solutions, subcircuit library

- **AICircuit** — SPICE-accurate LNA/PA/VCO/mixer/TX/RX dataset (NeurIPS 2024)
- **Steer RF Design** — 5-volume free textbook series (NC State, CC BY-NC)
- **Mendeley Antenna Dataset** — 55,000+ parametric patch antenna simulations
- **ReaLLMASIC/gLayout** — LLM-to-layout training pairs (exactly what we're building)

## Tier 2 — Satellite IC + Layout Code
- IEEE open Ka-band phased-array papers (radiation-hardened, LEO)
- OpenFASOC, ALIGN, gdsfactory, sky130_rf_tools, gLayout-IHP130
- CircuitNet 3.0 (general chip design dataset)

## Tier 3 — Broad EE Base
- HuggingFace: EEE-Bench, ElectricalNER, STEM-AI-mtl/Electrical-engineering
- Balanis Antenna Theory (Internet Archive)
- arXiv surrogate modeling papers

Output: `data/raw/` — raw text files, one per source

In [1]:
!pip install pymupdf arxiv requests tqdm gitpython datasets huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.19-py310-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 6.2 MB/s  0:00:04m0:00:0100:01
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Using cached smmap-5.0.3-py3-none-any.whl (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 24.0 MB/s  0:00:00
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
Using cached multiprocess-0.70.19-py310-none-any.whl (134 kB)
  Created wheel for sgmllib3k:

In [1]:
import os, json, re, subprocess, urllib.request, time
from pathlib import Path
from tqdm import tqdm

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1. Free Open-Access Textbooks — Direct Download

In [2]:
import requests, time
from pathlib import Path

BOOK_DIR = RAW_DIR / 'books'
BOOK_DIR.mkdir(exist_ok=True)

def _download(url: str, dest: Path, label: str, timeout: int = 60) -> bool:
    """Download with browser-like headers so repository redirects resolve properly."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36',
        'Accept': 'application/pdf,*/*',
    }
    try:
        r = requests.get(url, headers=headers, stream=True, timeout=timeout, allow_redirects=True)
        r.raise_for_status()
        content_type = r.headers.get('Content-Type', '')
        size = 0
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=65536):
                f.write(chunk)
                size += len(chunk)
        # Reject HTML "error pages" saved as .pdf (< 50 KB is suspicious)
        if size < 50_000 and 'pdf' not in content_type.lower():
            dest.unlink(missing_ok=True)
            print(f'    FAILED (got {size/1e3:.0f} KB, Content-Type: {content_type})')
            return False
        print(f'    OK  {size/1e6:.1f} MB')
        return True
    except Exception as e:
        dest.unlink(missing_ok=True)
        print(f'    FAILED: {e}')
        return False

# ── NC State Microwave and RF Design series (CC BY-NC-SA) ────────────────────
# Corrected direct-download bitstream URLs (v6 edition, 2022)
STEER_BOOKS = {
    'steer_v1_radio_systems.pdf':
        'https://repository.lib.ncsu.edu/bitstreams/8a9c254b-ecb0-40e3-87b0-d1c262eabcce/download',
    'steer_v2_transmission_lines.pdf':
        'https://repository.lib.ncsu.edu/bitstreams/338f7704-c1d5-4803-8cf4-c24f1af85fc6/download',
    'steer_v3_networks.pdf':
        'https://repository.lib.ncsu.edu/bitstreams/b00b0014-dc53-4976-ad25-728362afc8e3/download',
    'steer_v4_microwave_components.pdf':
        'https://repository.lib.ncsu.edu/bitstreams/a20b497c-5449-44ad-a010-80b1becbe5ec/download',
    'steer_v5_amplifiers_oscillators.pdf':
        'https://repository.lib.ncsu.edu/server/api/core/bitstreams/31bdf970-5583-4296-900c-fd8483bb4ad2/content',
}

# ── 5 additional open-access books ───────────────────────────────────────────
EXTRA_BOOKS = {
    # Steer Vol 0: Fundamentals of Microwave and RF Design (companion intro volume)
    'steer_v0_fundamentals.pdf':
        'https://repository.lib.ncsu.edu/bitstreams/a20b497c-5449-44ad-a010-80b1becbe5ec/download',

    # Orfanidis — Electromagnetic Waves and Antennas (Rutgers, author-posted free, ~1100 pp)
    # Covers EM theory, transmission lines, waveguides, antennas, propagation, radar
    'orfanidis_electromagnetic_waves_antennas.pdf':
        'https://eceweb1.rutgers.edu/~orfanidi/ewa/ewa-1up.pdf',

    # MIT 6.013 — Electromagnetics and Applications (MIT OCW, CC-BY-NC-SA)
    # Full course notes: Maxwell, plane waves, transmission lines, waveguides, antennas
    'mit6013_electromagnetics_applications.pdf':
        'https://ocw.mit.edu/courses/6-013-electromagnetics-and-applications-spring-2009/'
        'd3be4ea78b036a6362230fb41780cf54_MIT6_013S09_notes.pdf',

    # Chris Bowick — RF Circuit Design 2nd Ed (Internet Archive open-access scan)
    # Practical LNA/PA/filter/matching network design with worked examples
    'bowick_rf_circuit_design_2e.pdf':
        'https://ia601604.us.archive.org/21/items/RFCircuitDesign2ndEdition/'
        'RF%20Circuit%20Design%20-%202nd%20Edition.pdf',

    # NIST Technical Note — Fundamentals of RF and Microwave Power Measurements (gov, public domain)
    # Calibration, S-parameter, noise figure measurement methodology
    'nist_rf_microwave_power_measurements.pdf':
        'https://tsapps.nist.gov/publication/get_pdf.cfm?pub_id=921024',
}

ALL_BOOKS = {**STEER_BOOKS, **EXTRA_BOOKS}

print(f'Downloading {len(ALL_BOOKS)} RF textbooks...\n')
for fname, url in ALL_BOOKS.items():
    out_path = BOOK_DIR / fname
    if out_path.exists():
        print(f'  [skip] {fname}  ({out_path.stat().st_size/1e6:.1f} MB already)')
        continue
    print(f'  {fname}')
    ok = _download(url, out_path, fname)
    if not ok:
        print(f'    → Manual download link: {url}')
    time.sleep(1.0)   # be polite to servers

print('\nAlso place these copyrighted books manually in data/raw/books/ if you have them:')
MANUAL_BOOKS = [
    'pozar_microwave_engineering_4e.pdf        (Pozar, Wiley 2011)',
    'balanis_antenna_theory_4e.pdf             (Balanis, Wiley 2016)',
    'razavi_rf_microelectronics_2e.pdf         (Razavi, Prentice Hall 2012)',
    'gonzalez_microwave_transistor.pdf         (Gonzalez, Prentice Hall 1997)',
    'maas_nonlinear_microwave_circuits.pdf     (Maas, Artech House 2003)',
    'skolnik_radar_handbook_3e.pdf             (Skolnik, McGraw-Hill 2008)',
    'pratt_satellite_comms_3e.pdf              (Pratt et al., Wiley 2019)',
    'chahat_cubesat_antenna_design.pdf         (Chahat, NASA JPL / Wiley 2020)',
    'pozar_microwave_circuit_design.pdf        (Pozar, Wiley 2004)',
    'ludwig_bretchko_rf_circuit_design.pdf     (Ludwig & Bretchko, Pearson 2000)',
]
for b in MANUAL_BOOKS:
    exists = (BOOK_DIR / b.split()[0]).exists()
    status = '✓ found' if exists else '  missing'
    print(f'  {status}  {b}')



  [skip] steer_v1_radio_systems.pdf  (0.3 MB already)
  [skip] steer_v2_transmission_lines.pdf  (0.3 MB already)
  [skip] steer_v3_networks.pdf  (0.1 MB already)
  [skip] steer_v4_microwave_components.pdf  (18.9 MB already)
  [skip] steer_v5_amplifiers_oscillators.pdf  (13.0 MB already)
  [skip] steer_v0_fundamentals.pdf  (18.9 MB already)
  orfanidis_electromagnetic_waves_antennas.pdf
    FAILED: 403 Client Error: Forbidden for url: https://eceweb1.rutgers.edu/~orfanidi/ewa/ewa-1up.pdf
    → Manual download link: https://eceweb1.rutgers.edu/~orfanidi/ewa/ewa-1up.pdf
  [skip] mit6013_electromagnetics_applications.pdf  (5.6 MB already)
  [skip] bowick_rf_circuit_design_2e.pdf  (12.2 MB already)
  [skip] nist_rf_microwave_power_measurements.pdf  (0.7 MB already)

Also place these copyrighted books manually in data/raw/books/ if you have them:
    missing  pozar_microwave_engineering_4e.pdf        (Pozar, Wiley 2011)
    missing  balanis_antenna_theory_4e.pdf             (Balanis, Wiley 

In [3]:
import fitz  # pymupdf

TEXT_DIR = RAW_DIR / 'texts'
TEXT_DIR.mkdir(exist_ok=True)

def extract_pdf(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = page.get_text('text')
        lines = [l for l in text.split('\n') if len(l.strip()) > 3]
        pages.append('\n'.join(lines))
    return '\n\n--- PAGE BREAK ---\n\n'.join(pages)

# Extract all PDFs from books/ and pdfs/
pdf_sources = list((RAW_DIR / 'books').glob('*.pdf')) + list((RAW_DIR / 'pdfs').glob('*.pdf'))
for pdf_path in tqdm(pdf_sources, desc='Extracting PDFs'):
    out_path = TEXT_DIR / (pdf_path.stem + '.txt')
    if not out_path.exists():
        try:
            text = extract_pdf(pdf_path)
            out_path.write_text(text, encoding='utf-8')
            print(f'  {pdf_path.name}: {len(text):,} chars')
        except Exception as e:
            print(f'  FAILED {pdf_path.name}: {e}')

Extracting PDFs: 100%|██████████| 9/9 [00:00<00:00, 21694.68it/s]


## 2. Download arXiv Papers — Targeted RF IC / Phased Array / Satellite

In [ ]:
import arxiv

ARXIV_DIR = RAW_DIR / 'arxiv'
ARXIV_TEXT_DIR = ARXIV_DIR / 'texts'
ARXIV_PDF_DIR = ARXIV_DIR / 'pdfs'
for d in [ARXIV_DIR, ARXIV_TEXT_DIR, ARXIV_PDF_DIR]:
    d.mkdir(exist_ok=True)

# Specific high-value papers found by web search
SPECIFIC_ARXIV_IDS = [
    '2407.18272',  # AICircuit: RF circuit dataset (NeurIPS 2024)
    '2501.11839',  # Supervised learning for analog/RF circuit design benchmark
    '2008.10682',  # ALIGN: analog layout from SPICE netlists (DARPA)
    '2508.16403',  # Fast RFIC performance prediction via GNN
    '2508.10713',  # GPU-accelerated antenna EM simulation for ML
]

# Keyword queries for broader sweep
QUERIES = [
    'Ka-band CMOS phased array beamforming chip satellite',
    'millimeter wave low noise amplifier CMOS satellite LEO',
    'radiation hardened RF IC small satellite CubeSat',
    'CMOS Ka-band transceiver phased array integrated circuit',
    'RF CMOS LNA phase shifter variable gain amplifier layout',
    'gdsfactory glayout analog layout automation Python',
    'physics informed neural network electromagnetic antenna surrogate',
    'deep learning RF circuit design optimization S-parameter',
    'large language model analog IC circuit design automation',
    'neural network microstrip patch antenna geometry performance',
]

client = arxiv.Client()
papers_meta = []

# Fetch specific known papers first
for arxiv_id in SPECIFIC_ARXIV_IDS:
    search = arxiv.Search(id_list=[arxiv_id])
    for result in client.results(search):
        papers_meta.append({
            'id': result.entry_id, 'title': result.title,
            'abstract': result.summary, 'authors': [a.name for a in result.authors],
            'pdf_url': result.pdf_url, 'priority': 'high',
        })

# Keyword sweep
for query in tqdm(QUERIES, desc='arXiv queries'):
    search = arxiv.Search(query=query, max_results=15, sort_by=arxiv.SortCriterion.Relevance)
    for result in client.results(search):
        papers_meta.append({
            'id': result.entry_id, 'title': result.title,
            'abstract': result.summary, 'authors': [a.name for a in result.authors],
            'pdf_url': result.pdf_url, 'priority': 'normal', 'query': query,
        })

# Deduplicate
seen, unique_papers = set(), []
for p in papers_meta:
    if p['id'] not in seen:
        seen.add(p['id'])
        unique_papers.append(p)

print(f'Found {len(unique_papers)} unique papers')
(ARXIV_DIR / 'papers_meta.json').write_text(json.dumps(unique_papers, indent=2))

# Save abstracts corpus
abstracts = '\n\n'.join(
    f"# {p['title']}\nAuthors: {', '.join(p['authors'][:3])}\n\n{p['abstract']}"
    for p in unique_papers
)
(ARXIV_DIR / 'all_abstracts.txt').write_text(abstracts)

In [5]:
# Download full PDFs (high-priority first, then top-N by relevance)
high_priority = [p for p in unique_papers if p.get('priority') == 'high']
normal = [p for p in unique_papers if p.get('priority') != 'high']
download_list = high_priority + normal[:45]  # ~50 total

for paper in tqdm(download_list, desc='Downloading PDFs'):
    arxiv_id = paper['id'].split('/')[-1].replace('v1','').replace('v2','')
    out_pdf = ARXIV_PDF_DIR / f'{arxiv_id}.pdf'
    if not out_pdf.exists():
        try:
            urllib.request.urlretrieve(paper['pdf_url'], out_pdf)
            time.sleep(1.5)
        except Exception as e:
            print(f'  Failed {arxiv_id}: {e}')

for pdf_path in tqdm(list(ARXIV_PDF_DIR.glob('*.pdf')), desc='Extracting arXiv PDFs'):
    out_path = ARXIV_TEXT_DIR / (pdf_path.stem + '.txt')
    if not out_path.exists():
        try:
            out_path.write_text(extract_pdf(pdf_path), encoding='utf-8')
        except Exception as e:
            print(f'  {pdf_path.name}: {e}')

Extracting arXiv PDFs: 100%|██████████| 50/50 [00:05<00:00,  9.06it/s]


## 3. Clone GLayout / gdsfactory / RF Layout Repos

In [ ]:
CODE_DIR = RAW_DIR / 'code'
CODE_DIR.mkdir(exist_ok=True)

REPOS = [
    # Tier 1: LLM-to-layout — directly relevant to this project
    ('https://github.com/ReaLLMASIC/gLayout', 'glayout_llm'),
    # Tier 1: Open RF circuit layout generators
    ('https://github.com/idea-fasoc/OpenFASOC', 'openfasoc'),
    # Tier 1: Analog layout from SPICE (DARPA-funded, RF-aware)
    ('https://github.com/ALIGN-analoglayout/ALIGN-public', 'align_layout'),
    # Tier 2: Core GDS layout engine (used in this project)
    ('https://github.com/gdsfactory/gdsfactory', 'gdsfactory'),
    # Tier 2: RF design specifically with SKY130 open PDK
    ('https://github.com/diadatp/sky130_rf_tools', 'sky130_rf'),
    # Tier 2: GLayout for IHP 130nm BiCMOS (HBT, relevant for Ka-band)
    ('https://github.com/amisapta15/gLayout-IHP130', 'glayout_ihp130'),
    # Tier 2: scikit-rf — RF engineering Python library (huge codebase of RF math)
    ('https://github.com/scikit-rf/scikit-rf', 'skrf'),
    # Tier 2: openEMS — open FDTD EM solver (used for generating sim datasets)
    ('https://github.com/thliebig/openEMS', 'openems'),
    # Tier 3: AICircuit benchmark code
    ('https://github.com/AvestimehrResearchGroup/AICircuit', 'aicircuit'),
    # Tier 1: Chipster — user's own PySpice analog circuit generator + notebooks
    ('https://github.com/adeirman46/chipster', 'chipster'),
    # Tier 1: AnalogCoder — 24 analog circuit benchmark problems, PySpice solutions (AAAI 2025)
    ('https://github.com/laiyao1/AnalogCoder', 'analogcoder'),
]

for url, name in REPOS:
    dest = CODE_DIR / name
    if not dest.exists():
        print(f'Cloning {name}...')
        r = subprocess.run(['git', 'clone', '--depth=1', url, str(dest)],
                           capture_output=True, text=True)
        print(f'  {"OK" if r.returncode == 0 else "FAILED: " + r.stderr[:120]}')
    else:
        print(f'  {name} already exists')

In [7]:
CODE_TEXT_DIR = RAW_DIR / 'code_texts'
CODE_TEXT_DIR.mkdir(exist_ok=True)

RF_KEYWORDS = [
    'lna', 'amplifier', 'mixer', 'oscillator', 'vco', 'pll', 'filter',
    'antenna', 'patch', 'dipole', 'phased_array', 'beamform', 'phase_shift',
    'transmission_line', 'microstrip', 'coplanar', 'cpw', 'sparameter',
    'noise_figure', 'matching', 'impedance', 'resonator', 'balun', 'coupler',
    'transceiver', 'rf', 'microwave', 'mmwave', 'satellite', 'ka_band',
    'glayout', 'gdsfactory', 'layout', 'gds', 'pdk', 'inductor', 'capacitor',
]

all_py = list(CODE_DIR.rglob('*.py'))
relevant = [f for f in all_py if any(
    kw in f.read_text(errors='ignore').lower() for kw in RF_KEYWORDS
) if f.stat().st_size < 200_000]  # Skip giant auto-generated files

print(f'{len(relevant)} RF-relevant Python files out of {len(all_py)} total')

corpus_chunks = []
for py_file in relevant:
    try:
        content = py_file.read_text(errors='ignore')
        rel_path = py_file.relative_to(CODE_DIR)
        corpus_chunks.append(f'# === {rel_path} ===\n{content}')
    except Exception:
        pass

(CODE_TEXT_DIR / 'rf_code_corpus.py').write_text('\n\n'.join(corpus_chunks))
print(f'Code corpus: {sum(len(c) for c in corpus_chunks)/1e6:.1f} MB')

1203 RF-relevant Python files out of 1609 total
Code corpus: 8.7 MB


## 4. HuggingFace Datasets

In [3]:
from datasets import load_dataset
import json

HF_DIR = RAW_DIR / 'huggingface'
HF_DIR.mkdir(exist_ok=True)

# Confirmed non-empty HF datasets for RF/analog circuit finetuning
# (verified via HuggingFace Hub — all have actual rows)
HF_DATASETS = [
    # 1. MMLU Electrical Engineering — 285 EE multiple-choice questions (benchmark)
    ('cais/mmlu', 'electrical_engineering', {'split': 'test'}),

    # 2. MMLU-Pro Engineering — 969 harder engineering Q&A with chain-of-thought
    ('TIGER-Lab/MMLU-Pro', None, {'split': 'test'}),

    # 3. STEM-AI EE — 1,131 EE Q&A pairs covering circuits, signals, systems
    ('STEM-AI-mtl/Electrical-engineering', None, {'split': 'train'}),

    # 4. CAMEL Physics — 20,000 problem-solution pairs covering EM, optics, mechanics
    ('camel-ai/physics', None, {'split': 'train'}),

    # 5. CircuitSense — multimodal circuit perception/analysis/design Q&A
    ('armanakbari4/CircuitSense', None, {'split': 'train'}),

    # 6. Circuit Analysis Reasoning — KVL/KCL/Thevenin chain-of-thought, LaTeX solutions
    ('EngineeringWays/Circuit-Analysis-Reasoning-Sample', None, {'split': 'train'}),

    # 7. Open Schematics — 84,470 KiCad schematics with component metadata
    ('bshada/open-schematics', None, {'split': 'train'}),

    # 8. EEE-Bench — 2,860 multimodal EE problems (analog, EM, signal processing)
    ('afdsafas/EEE-Bench', None, {'split': 'test'}),
]

def _save_dataset(ds, out_path: Path):
    """Save dataset rows to JSONL, serializing image fields as metadata only."""
    count = 0
    with open(out_path, 'w') as f:
        for row in ds:
            # Convert row to plain dict; skip binary image bytes to keep JSONL readable
            clean = {}
            for k, v in row.items():
                if hasattr(v, 'tobytes') or (hasattr(v, '__class__') and
                   v.__class__.__name__ in ('Image', 'PIL.Image.Image')):
                    clean[k] = f'<image {getattr(v, "size", "?")}>'
                elif isinstance(v, bytes):
                    clean[k] = f'<bytes {len(v)}>'
                else:
                    try:
                        json.dumps(v)   # test serialisability
                        clean[k] = v
                    except (TypeError, ValueError):
                        clean[k] = str(v)
            f.write(json.dumps(clean) + '\n')
            count += 1
    return count

for entry in HF_DATASETS:
    ds_name, config, kwargs = entry
    safe_name = ds_name.replace('/', '_') + (f'_{config}' if config else '')
    out_path  = HF_DIR / (safe_name + '.jsonl')

    if out_path.exists():
        n = sum(1 for _ in open(out_path))
        print(f'  [skip] {ds_name}  ({n:,} rows already)')
        continue

    print(f'Loading {ds_name}' + (f' [{config}]' if config else '') + '...')
    try:
        if config:
            ds = load_dataset(ds_name, config, **kwargs)
        else:
            ds = load_dataset(ds_name, **kwargs)

        # MMLU-Pro: filter to engineering category only
        if ds_name == 'TIGER-Lab/MMLU-Pro':
            ds = ds.filter(lambda x: x.get('category', '').lower() == 'engineering')

        n = _save_dataset(ds, out_path)
        print(f'  Saved {n:,} rows  ->  {out_path.name}')
    except Exception as e:
        print(f'  FAILED: {e}')

print('\nHuggingFace datasets summary:')
for f in sorted(HF_DIR.glob('*.jsonl')):
    n = sum(1 for _ in open(f))
    print(f'  {n:6,} rows  {f.name}')


  [skip] cais/mmlu  (145 rows already)
  [skip] TIGER-Lab/MMLU-Pro  (969 rows already)
  [skip] STEM-AI-mtl/Electrical-engineering  (1,131 rows already)
  [skip] camel-ai/physics  (20,000 rows already)
Loading armanakbari4/CircuitSense...


Resolving data files:   0%|          | 0/42636 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

  FAILED: An error occurred while generating the dataset
Loading EngineeringWays/Circuit-Analysis-Reasoning-Sample...


README.md: 0.00B [00:00, ?B/s]

...

data/train-00076-of-00078.parquet:   0%|          | 0.00/6.47M [00:00<?, ?B/s]

data/train-00077-of-00078.parquet:   0%|          | 0.00/11.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84470 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

  FAILED: cannot identify image file <_io.BytesIO object at 0x7d55186812b0>
Loading afdsafas/EEE-Bench...
  Saved 2,860 rows  ->  afdsafas_EEE-Bench.jsonl

HuggingFace datasets summary:
      50 rows  EngineeringWays_Circuit-Analysis-Reasoning-Sample.jsonl
   1,131 rows  STEM-AI-mtl_Electrical-engineering.jsonl
     969 rows  TIGER-Lab_MMLU-Pro.jsonl
   2,860 rows  afdsafas_EEE-Bench.jsonl
     314 rows  bshada_open-schematics.jsonl
     145 rows  cais_mmlu_electrical_engineering.jsonl
  20,000 rows  camel-ai_physics.jsonl


## 5. AICircuit Dataset — RF Circuit SPICE Simulations (NeurIPS 2024)

In [2]:
# AICircuit is the most important RF dataset — clone separately and inspect
AICIRCUIT_DIR = RAW_DIR / 'aicircuit'
if not AICIRCUIT_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth=1',
        'https://github.com/AvestimehrResearchGroup/AICircuit',
        str(AICIRCUIT_DIR)
    ])

# List available circuit datasets
for f in sorted(AICIRCUIT_DIR.rglob('*.csv'))[:20]:
    print(f'  {f.relative_to(AICIRCUIT_DIR)}  ({f.stat().st_size/1e3:.0f} KB)')

Cloning into '../data/raw/aicircuit'...


  Dataset/CSVA/CSVA.csv  (355 KB)
  Dataset/CVA/CVA.csv  (771 KB)
  Dataset/LNA/LNA.csv  (2813 KB)
  Dataset/Mixer/Mixer.csv  (913 KB)
  Dataset/PA/PA.csv  (558 KB)
  Dataset/Receiver/Receiver.csv  (23038 KB)
  Dataset/TSVA/TSVA.csv  (1357 KB)
  Dataset/Transmitter/Transmitter.csv  (14914 KB)
  Dataset/VCO/VCO.csv  (1299 KB)


In [3]:
import pandas as pd

# Load and preview each circuit dataset
circuit_dfs = {}
for csv_file in sorted(AICIRCUIT_DIR.rglob('*.csv')):
    try:
        df = pd.read_csv(csv_file)
        circuit_name = csv_file.stem
        circuit_dfs[circuit_name] = df
        print(f'\n{circuit_name}: {df.shape[0]} samples, {df.shape[1]} features')
        print(f'  Columns: {list(df.columns[:8])}...')
    except Exception as e:
        print(f'  {csv_file.name}: {e}')


CSVA: 7840 samples, 7 features
  Columns: ['VDD', 'Vgate', 'Wn', 'Rd', 'Bandwidth', 'PowerConsumption', 'VoltageGain']...

CVA: 15079 samples, 7 features
  Columns: ['Wbias', 'Rd', 'Wn1', 'Wn2', 'Bandwidth', 'PowerConsumption', 'VoltageGain']...

LNA: 31993 samples, 12 features
  Columns: ['C1', 'C2', 'Ld', 'Lg', 'Ls', 'WN1', 'WN2', 'GTMax']...

Mixer: 17149 samples, 8 features
  Columns: ['C', 'R', 'WLO', 'WRF', 'PowerConsumption', 'IFVoltageSwing', 'ConversionGain', 'NoiseFigure']...

PA: 5584 samples, 14 features
  Columns: ['L1p', 'L1s', 'L3p', 'L3s', 'Lm', 'Wn1', 'Wn2', 'S11']...

Receiver: 155368 samples, 23 features
  Columns: ['C1_LNA', 'C2_LNA', 'Ld_LNA', 'Lg_LNA', 'Ls_LNA', 'WN1_LNA', 'WN2_LNA', 'C_Mixer']...

TSVA: 19290 samples, 9 features
  Columns: ['C1', 'WNOUT', 'WPOUT', 'WCS', 'WN1', 'WP1', 'Bandwidth', 'PowerConsumption']...

Transmitter: 95346 samples, 21 features
  Columns: ['C1_VCO', 'L1_VCO', 'Rp_VCO', 'Wn_VCO', 'Wn6_VCO', 'Wvar_VCO', 'L1p_PA', 'L1s_PA']...

VCO:

## 6. Mendeley Antenna Dataset — 55,000+ Parametric Simulations

In [4]:
# Mendeley dataset: 55K parametric patch antenna simulations
# URL: https://data.mendeley.com/datasets/3gxr2vvd9n/2
# Must be downloaded manually (requires Mendeley account or direct link)
ANTENNA_DIR = RAW_DIR / 'antenna_dataset'
ANTENNA_DIR.mkdir(exist_ok=True)

print('Mendeley Antenna Dataset (55K samples):')
print('  URL: https://data.mendeley.com/datasets/3gxr2vvd9n/2')
print('  Contains: substrate W/L/H, patch W/L, feedline W, slot dims -> gain, S11, efficiency')
print(f'  Place downloaded CSV/XLSX in: {ANTENNA_DIR}')

# Try to load if already downloaded
for f in ANTENNA_DIR.glob('*.csv'):
    df = pd.read_csv(f)
    print(f'\nLoaded {f.name}: {df.shape}')
    print(df.head(3))

Mendeley Antenna Dataset (55K samples):
  URL: https://data.mendeley.com/datasets/3gxr2vvd9n/2
  Contains: substrate W/L/H, patch W/L, feedline W, slot dims -> gain, S11, efficiency
  Place downloaded CSV/XLSX in: ../data/raw/antenna_dataset


## 7. Chipster + AnalogCoder — Analog PySpice Circuit Datasets

Two PySpice-based analog circuit sources:

- **Chipster** (`adeirman46/chipster`) — PySpice analog generator notebooks + `src/analog_generator/` code  
- **AnalogCoder** (`laiyao1/AnalogCoder`, AAAI 2025 Oral) — 24 circuit design problems, PySpice sample solutions, subcircuit library

Extracts both into:
- `data/raw/analog_pyspice/corpus.py` — all PySpice code concatenated (for pre-training)
- `data/raw/analog_pyspice/sft_pairs.jsonl` — instruction→PySpice code pairs (for SFT)


In [5]:
import subprocess, json, csv
from pathlib import Path

ANALOG_DIR = RAW_DIR / 'analog_pyspice'
ANALOG_DIR.mkdir(exist_ok=True)

CODE_DIR = RAW_DIR / 'code'
CODE_DIR.mkdir(exist_ok=True)
CHIPSTER_DIR  = CODE_DIR / 'chipster'
ANALOGCODER_DIR = CODE_DIR / 'analogcoder'

# ── Clone if missing (repos also added to main REPOS list above) ─────────────
for url, dest in [
    ('https://github.com/adeirman46/chipster',   CHIPSTER_DIR),
    ('https://github.com/laiyao1/AnalogCoder',   ANALOGCODER_DIR),
]:
    if not dest.exists():
        print(f'Cloning {dest.name}...')
        r = subprocess.run(['git', 'clone', '--depth=1', url, str(dest)],
                           capture_output=True, text=True)
        print(f'  {"OK" if r.returncode == 0 else "FAILED: " + r.stderr[:200]}')
    else:
        print(f'  {dest.name} already cloned')

print()

# ════════════════════════════════════════════════════════════════════════════
# A. Extract PySpice / Python code from Chipster
# ════════════════════════════════════════════════════════════════════════════
chipster_chunks = []

PYSPICE_KEYWORDS = [
    'pyspice', 'spice', 'circuit()', 'Circuit()', 'ngspice',
    'nmos', 'pmos', 'mosfet', 'transistor', 'resistor', 'capacitor',
    'inductor', 'voltage_source', 'current_source', 'simulation',
    'operating_point', 'transient', 'ac', 'dc', 'subcircuit',
    'analog', 'amplifier', 'opamp', 'lna', 'mixer', 'vco',
]

def is_pyspice_relevant(text: str) -> bool:
    low = text.lower()
    return any(kw in low for kw in PYSPICE_KEYWORDS)

# Collect .py files from chipster
if CHIPSTER_DIR.exists():
    py_files = list(CHIPSTER_DIR.rglob('*.py'))
    for f in py_files:
        try:
            content = f.read_text(errors='ignore')
            if is_pyspice_relevant(content) and len(content) > 100:
                rel = f.relative_to(CHIPSTER_DIR)
                chipster_chunks.append(f'# === chipster/{rel} ===\n{content}')
        except Exception:
            pass

    # Also extract code cells from Jupyter notebooks
    import json as _json
    for nb_file in CHIPSTER_DIR.rglob('*.ipynb'):
        try:
            nb_data = _json.loads(nb_file.read_text(errors='ignore'))
            cell_codes = []
            for cell in nb_data.get('cells', []):
                if cell.get('cell_type') == 'code':
                    src = ''.join(cell.get('source', []))
                    if is_pyspice_relevant(src) and len(src.strip()) > 50:
                        cell_codes.append(src)
            if cell_codes:
                rel = nb_file.relative_to(CHIPSTER_DIR)
                combined = f'# === chipster/{rel} ===\n' + '\n# ---\n'.join(cell_codes)
                chipster_chunks.append(combined)
        except Exception:
            pass

print(f'Chipster: {len(chipster_chunks)} PySpice code chunks')

# ════════════════════════════════════════════════════════════════════════════
# B. Extract AnalogCoder data: problem_set.tsv + sample_design/ + subcircuit_lib/
# ════════════════════════════════════════════════════════════════════════════
analogcoder_chunks = []
sft_pairs = []   # {"instruction": ..., "output": ...}

if ANALOGCODER_DIR.exists():
    # ── B1. Read problem_set.tsv ─────────────────────────────────────────────
    problems = {}
    tsv_path = ANALOGCODER_DIR / 'problem_set.tsv'
    if tsv_path.exists():
        with open(tsv_path, newline='') as fh:
            reader = csv.DictReader(fh, delimiter='\t')
            for row in reader:
                name = row.get('name', row.get('circuit', '')).strip()
                if name:
                    problems[name] = row

    print(f'AnalogCoder: {len(problems)} problem definitions in problem_set.tsv')

    # ── B2. Read lib_info.tsv ─────────────────────────────────────────────────
    lib_info_path = ANALOGCODER_DIR / 'lib_info.tsv'
    lib_text = ''
    if lib_info_path.exists():
        lib_text = lib_info_path.read_text(errors='ignore')
        analogcoder_chunks.append('# === AnalogCoder/lib_info.tsv ===\n' + lib_text)

    # ── B3. Subcircuit library ────────────────────────────────────────────────
    subcirc_dir = ANALOGCODER_DIR / 'subcircuit_lib'
    if subcirc_dir.exists():
        for f in sorted(subcirc_dir.rglob('*.py')):
            content = f.read_text(errors='ignore')
            if len(content.strip()) > 50:
                rel = f.relative_to(ANALOGCODER_DIR)
                analogcoder_chunks.append(f'# === analogcoder/{rel} ===\n{content}')

    # ── B4. Sample designs → SFT pairs ───────────────────────────────────────
    sample_dir = ANALOGCODER_DIR / 'sample_design'
    if not sample_dir.exists():
        # Some repos use different directory names
        for candidate in ['sample_design', 'samples', 'designs', 'solutions']:
            p = ANALOGCODER_DIR / candidate
            if p.exists():
                sample_dir = p
                break

    if sample_dir.exists():
        for py_file in sorted(sample_dir.rglob('*.py')):
            try:
                code = py_file.read_text(errors='ignore').strip()
                if len(code) < 80:
                    continue
                circuit_name = py_file.stem  # e.g. "TwoStageOpampMiller"
                rel = py_file.relative_to(ANALOGCODER_DIR)

                # Build human-readable instruction from problem_set if available
                prob = problems.get(circuit_name, {})
                if prob:
                    # Build spec string from TSV columns
                    spec_parts = []
                    for k, v in prob.items():
                        if v and k not in ('name', 'circuit'):
                            spec_parts.append(f'{k}: {v}')
                    spec_str = '\n'.join(spec_parts) if spec_parts else ''
                    instruction = (
                        f'Design a {circuit_name} analog circuit using PySpice.\n'
                        + (f'Specifications:\n{spec_str}\n' if spec_str else '')
                        + 'Write complete runnable PySpice Python code including netlist, '
                        'simulation setup, and performance extraction.'
                    )
                else:
                    instruction = (
                        f'Design a {circuit_name} analog circuit using PySpice. '
                        'Write complete runnable Python code with the full circuit netlist, '
                        'ngspice simulation, and measurement of key performance metrics.'
                    )

                sft_pairs.append({
                    'source': 'analogcoder',
                    'circuit': circuit_name,
                    'instruction': instruction,
                    'input': '',
                    'output': code,
                })
                analogcoder_chunks.append(f'# === analogcoder/{rel} ===\n{code}')
            except Exception as e:
                print(f'  {py_file.name}: {e}')

    # ── B5. problem_check/ testbenches → pre-train corpus ─────────────────────
    check_dir = ANALOGCODER_DIR / 'problem_check'
    if check_dir.exists():
        for f in sorted(check_dir.rglob('*.py')):
            content = f.read_text(errors='ignore')
            if len(content.strip()) > 50:
                rel = f.relative_to(ANALOGCODER_DIR)
                analogcoder_chunks.append(f'# === analogcoder/{rel} ===\n{content}')

    print(f'AnalogCoder: {len(analogcoder_chunks)} code chunks, {len(sft_pairs)} SFT pairs from sample_design/')

# ════════════════════════════════════════════════════════════════════════════
# C. Build SFT pairs from Chipster notebooks (code → instruction)
# ════════════════════════════════════════════════════════════════════════════
for chunk in chipster_chunks:
    lines = chunk.split('\n')
    header = lines[0]  # "# === chipster/path/to/file.py ==="
    code_body = '\n'.join(lines[1:]).strip()
    if len(code_body) < 100:
        continue

    # Derive circuit name from filename
    import re as _re
    fname_match = _re.search(r'chipster/(.+?)\.(?:py|ipynb)', header)
    circuit_hint = fname_match.group(1).split('/')[-1].replace('_', ' ') if fname_match else 'analog circuit'

    sft_pairs.append({
        'source': 'chipster',
        'circuit': circuit_hint,
        'instruction': (
            f'Write PySpice Python code for a {circuit_hint} analog circuit. '
            'Include the full netlist definition, simulation setup, and any '
            'performance metric extraction (gain, noise figure, bandwidth, etc.).'
        ),
        'input': '',
        'output': code_body[:4000],  # cap very long files
    })

print(f'\nTotal SFT pairs (chipster + analogcoder): {len(sft_pairs)}')

# ════════════════════════════════════════════════════════════════════════════
# D. Save outputs
# ════════════════════════════════════════════════════════════════════════════

# D1. Combined PySpice code corpus (for continued pre-training)
all_chunks = chipster_chunks + analogcoder_chunks
corpus_path = ANALOG_DIR / 'corpus.py'
corpus_path.write_text('\n\n'.join(all_chunks), encoding='utf-8')
print(f'\nPre-train corpus: {len(all_chunks)} chunks → {corpus_path.name} '
      f'({corpus_path.stat().st_size/1e3:.0f} KB)')

# D2. SFT pairs JSONL
sft_path = ANALOG_DIR / 'sft_pairs.jsonl'
with open(sft_path, 'w') as f:
    for pair in sft_pairs:
        f.write(json.dumps(pair) + '\n')
print(f'SFT pairs: {len(sft_pairs)} → {sft_path.name}')

# D3. Quick preview
print('\nSample SFT pair:')
if sft_pairs:
    p = sft_pairs[0]
    print(f'  circuit:     {p["circuit"]}')
    print(f'  source:      {p["source"]}')
    print(f'  instruction: {p["instruction"][:120]}...')
    print(f'  output:      {p["output"][:200]}...')

Cloning chipster...


  OK
Cloning analogcoder...
  OK

Chipster: 55 PySpice code chunks
AnalogCoder: 0 problem definitions in problem_set.tsv
AnalogCoder: 63 code chunks, 35 SFT pairs from sample_design/

Total SFT pairs (chipster + analogcoder): 90

Pre-train corpus: 118 chunks → corpus.py (908 KB)
SFT pairs: 90 → sft_pairs.jsonl

Sample SFT pair:
  circuit:     p1
  source:      analogcoder
  instruction: Design a p1 analog circuit using PySpice. Write complete runnable Python code with the full circuit netlist, ngspice sim...
  output:      from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *
circuit = Circuit('Single-Stage Common-Source Amplifier')
# Define the NMOS transistor model
circuit.model('nmos_model', 'nmos', le...


## Summary

In [6]:
print('=== Raw Data Summary ===')
for subdir in sorted(RAW_DIR.iterdir()):
    if subdir.is_dir():
        files = list(subdir.rglob('*'))
        size = sum(f.stat().st_size for f in files if f.is_file())
        print(f'  {subdir.name:25s}  {len(files):5d} files  {size/1e6:7.1f} MB')
print('\nNext: Run 02_dataset_build.ipynb to create SFT training data')

=== Raw Data Summary ===
  aicircuit                    386 files     55.7 MB
  analog_pyspice                 2 files      1.2 MB
  antenna_dataset                1 files      0.2 MB
  arxiv                        104 files    169.5 MB
  books                          9 files     70.0 MB
  code                        9664 files    998.8 MB
  code_texts                     1 files      8.7 MB
  huggingface                    7 files    142.9 MB
  texts                          9 files      3.3 MB

Next: Run 02_dataset_build.ipynb to create SFT training data
